In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, coalesce, lit, expr, split, lpad, concat

# =============================================================================
# COMPREHENSIVE DATA CLEANING: Handle NULL prices, negative qty, NULL discount
# =============================================================================

# ---------------------------------------------------
# 1. Read Real Tables from Bronze Catalog
# ---------------------------------------------------
invoice_line_items = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items")
invoices = spark.table("bronze_catalog.bronze_stage_schema.invoices")
payments = spark.table("bronze_catalog.bronze_stage_schema.payments")
products = spark.table("bronze_catalog.bronze_stage_schema.products")
exchange_rates = spark.table("bronze_catalog.bronze_raw_schema.exchange_rates")  # Note: in bronze_raw_schema

print("Tables loaded successfully")
print(f"Invoice Line Items: {invoice_line_items.count()} rows")
print(f"Invoices: {invoices.count()} rows")
print(f"Payments: {payments.count()} rows")
print(f"Products: {products.count()} rows")
print(f"Exchange Rates: {exchange_rates.count()} rows")

# ---------------------------------------------------
# 2. Join Products to Fix NULL unit_price
# ---------------------------------------------------
df = invoice_line_items.join(products, "product_id", "left")

# Use cost_price when unit_price is NULL
df = df.withColumn(
    "unit_price_clean",
    coalesce(col("unit_price"), col("cost_price"))
)

# ---------------------------------------------------
# 3. Keep Negative Quantity (Treat as Returns)
# ---------------------------------------------------
df = df.withColumn(
    "quantity_clean", 
    col("quantity")
)

# ---------------------------------------------------
# 4. Calculate Gross Amount
# ---------------------------------------------------
df = df.withColumn(
    "gross_amount",
    col("quantity_clean") * col("unit_price_clean")
)

# ---------------------------------------------------
# 5. Calculate Known Net Amount (for items with discount)
# ---------------------------------------------------
df = df.withColumn(
    "known_net_amount",
    when(col("discount").isNotNull(),
         col("gross_amount") * (1 - col("discount")))
)

# ---------------------------------------------------
# 6. Join Payment Data
# ---------------------------------------------------
df = df.join(
    payments.select("invoice_id", "payment_amount"),
    "invoice_id",
    "left"
)

# ---------------------------------------------------
# 7. Calculate Total Known Net Amount per Invoice
# ---------------------------------------------------
known_totals = df.groupBy("invoice_id").agg(
    spark_sum("known_net_amount").alias("total_known_net"),
    spark_sum("gross_amount").alias("total_gross")
)

df = df.join(known_totals, "invoice_id")

# ---------------------------------------------------
# 8. Calculate Remaining Payment to Allocate
# ---------------------------------------------------
df = df.withColumn(
    "remaining_amount",
    col("payment_amount") - coalesce(col("total_known_net"), lit(0))
)

# ---------------------------------------------------
# 9. Calculate Missing Discount from Payment Data
#    discount = 1 - (remaining_amount / gross_amount)
# ---------------------------------------------------
df = df.withColumn(
    "discount_clean",
    when(
        col("discount").isNull() & col("payment_amount").isNotNull(),
        1 - (col("remaining_amount") / col("gross_amount"))
    ).otherwise(coalesce(col("discount"), lit(0.0)))  # If no payment, assume 0 discount
)

# ---------------------------------------------------
# 10. Calculate Final Line Amount in Original Currency
# ---------------------------------------------------
df = df.withColumn(
    "line_amount_original_currency",
    col("gross_amount") * (1 - col("discount_clean"))
)

# ---------------------------------------------------
# 11. Join Invoice for Currency Information
# ---------------------------------------------------
df = df.join(
    invoices.select("invoice_id", col("currency").alias("invoice_currency"), "invoice_date"),
    "invoice_id",
    "left"
)

# ---------------------------------------------------
# 12. Join Exchange Rates for Currency Conversion
# ---------------------------------------------------
# Prepare exchange rates with date formatting
# Both exchange_rates.date and invoices.invoice_date use yyyy/M/d format - normalize both
exchange_rates_clean = exchange_rates.withColumn(
    "er_date_parts", split(col("date"), "/")
).withColumn(
    "date_normalized",
    concat(
        col("er_date_parts")[0],  # year
        lit("-"),
        lpad(col("er_date_parts")[1], 2, "0"),  # month with padding
        lit("-"),
        lpad(col("er_date_parts")[2], 2, "0")   # day with padding
    )
).withColumn(
    "date_clean",
    expr("to_date(date_normalized, 'yyyy-MM-dd')")
).select(
    col("date_clean"),
    col("currency").alias("exchange_currency"),
    col("rate_to_usd")
)

# Parse invoice_date - format is yyyy/M/d (e.g., 2023/1/8, 2024/12/31)
# Normalize by splitting and padding components, then reconstructing as yyyy-MM-dd
df = df.withColumn("inv_date_parts", split(col("invoice_date"), "/"))
df = df.withColumn(
    "invoice_date_normalized",
    concat(
        col("inv_date_parts")[0],  # year
        lit("-"),
        lpad(col("inv_date_parts")[1], 2, "0"),  # month with padding
        lit("-"),
        lpad(col("inv_date_parts")[2], 2, "0")   # day with padding
    )
)

df = df.withColumn(
    "invoice_date_clean",
    expr("to_date(invoice_date_normalized, 'yyyy-MM-dd')")
)

# Join exchange rates
df = df.join(
    exchange_rates_clean,
    (col("invoice_date_clean") == col("date_clean")) & (col("invoice_currency") == col("exchange_currency")),
    "left"
)

# ---------------------------------------------------
# 13. Convert to USD
# ---------------------------------------------------
df = df.withColumn(
    "line_amount_usd",
    col("line_amount_original_currency") * coalesce(col("rate_to_usd"), lit(1.0))  # Default to 1.0 if no rate
)

# ---------------------------------------------------
# 14. Add Data Quality Flags
# ---------------------------------------------------
df = df.withColumn(
    "unit_price_source",
    when(col("unit_price").isNull(), "cost_price").otherwise("original")
)

df = df.withColumn(
    "discount_source",
    when(col("discount").isNull() & col("payment_amount").isNotNull(), "payment_calculated")
    .when(col("discount").isNull(), "default_zero")
    .otherwise("original")
)

df = df.withColumn(
    "is_return",
    when(col("quantity_clean") < 0, True).otherwise(False)
)

df = df.withColumn(
    "exchange_rate_available",
    when(col("rate_to_usd").isNotNull(), True).otherwise(False)
)

# ---------------------------------------------------
# 15. Final Output
# ---------------------------------------------------
result = df.select(
    "invoice_line_id",
    "invoice_id",
    "product_id",
    col("product_name"),
    col("category"),
    col("quantity").alias("quantity_original"),
    col("quantity_clean"),
    col("unit_price").alias("unit_price_original"),
    col("unit_price_clean"),
    col("discount").alias("discount_original"),
    col("discount_clean"),
    "gross_amount",
    "line_amount_original_currency",
    col("invoice_currency").alias("currency"),
    "rate_to_usd",
    "line_amount_usd",
    "unit_price_source",
    "discount_source",
    "is_return",
    "exchange_rate_available"
)

print("\n" + "="*80)
print("DATA CLEANING COMPLETE")
print("="*80)

# Display sample results
display(result.orderBy("invoice_id", "invoice_line_id").limit(50))

In [0]:
# =============================================================================
# COMPREHENSIVE VALIDATION OF DATA CLEANING
# =============================================================================

from pyspark.sql.functions import count, avg, min, max, sum as spark_sum, countDistinct

print("="*80)
print("DATA CLEANING VALIDATION REPORT")
print("="*80)

# ---------------------------------------------------
# 1. NULL Unit Price Handling
# ---------------------------------------------------
print("\n1. NULL UNIT PRICE HANDLING")
print("-" * 80)

null_price_stats = result.groupBy("unit_price_source").agg(
    count("*").alias("count"),
    avg("unit_price_clean").alias("avg_price")
)

print("\nUnit Price Sources:")
display(null_price_stats)

print("\nSample of records where cost_price was used:")
display(result.filter(col("unit_price_source") == "cost_price").limit(10))

# ---------------------------------------------------
# 2. Negative Quantity (Returns)
# ---------------------------------------------------
print("\n2. NEGATIVE QUANTITY (RETURNS) ANALYSIS")
print("-" * 80)

return_stats = result.groupBy("is_return").agg(
    count("*").alias("count"),
    spark_sum("line_amount_usd").alias("total_usd")
)

print("\nReturn vs Regular Sales:")
display(return_stats)

print("\nSample of return transactions (negative quantity):")
display(result.filter(col("is_return") == True).limit(10))

# ---------------------------------------------------
# 3. Discount Calculation Validation
# ---------------------------------------------------
print("\n3. DISCOUNT CALCULATION VALIDATION")
print("-" * 80)

discount_stats = result.groupBy("discount_source").agg(
    count("*").alias("count"),
    avg("discount_clean").alias("avg_discount"),
    min("discount_clean").alias("min_discount"),
    max("discount_clean").alias("max_discount")
)

print("\nDiscount Sources:")
display(discount_stats)

# Check for invalid discounts (< 0 or > 1)
invalid_discounts = result.filter(
    (col("discount_clean") < 0) | (col("discount_clean") > 1)
)

print(f"\nInvalid Discounts (< 0 or > 1): {invalid_discounts.count()} records")
if invalid_discounts.count() > 0:
    print("\nSample of invalid discounts:")
    display(invalid_discounts.select(
        "invoice_line_id", "invoice_id", "product_id", 
        "gross_amount", "discount_clean", "discount_source"
    ).limit(10))

print("\nSample of payment-calculated discounts:")
display(result.filter(col("discount_source") == "payment_calculated").limit(10))

# ---------------------------------------------------
# 4. Currency Conversion Validation
# ---------------------------------------------------
print("\n4. CURRENCY CONVERSION VALIDATION")
print("-" * 80)

currency_coverage = result.groupBy("currency", "exchange_rate_available").agg(
    count("*").alias("count"),
    avg("rate_to_usd").alias("avg_rate")
)

print("\nExchange Rate Coverage by Currency:")
display(currency_coverage)

print("\nSample with USD conversion:")
display(result.select(
    "invoice_line_id", "currency", "line_amount_original_currency", 
    "rate_to_usd", "line_amount_usd"
).limit(10))

# ---------------------------------------------------
# 5. Overall Summary Statistics
# ---------------------------------------------------
print("\n5. OVERALL SUMMARY STATISTICS")
print("-" * 80)

total_records = result.count()
print(f"Total Records: {total_records:,}")

summary = result.select(
    spark_sum("line_amount_original_currency").alias("total_original_currency"),
    spark_sum("line_amount_usd").alias("total_usd"),
    avg("discount_clean").alias("avg_discount"),
    countDistinct("invoice_id").alias("unique_invoices"),
    countDistinct("product_id").alias("unique_products")
).collect()[0]

print(f"\nTotal Amount (Original Currency): {summary['total_original_currency']:,.2f}")
print(f"Total Amount (USD): {summary['total_usd']:,.2f}")
print(f"Average Discount: {summary['avg_discount']:.4f} ({summary['avg_discount']*100:.2f}%)")
print(f"Unique Invoices: {summary['unique_invoices']:,}")
print(f"Unique Products: {summary['unique_products']:,}")

# ---------------------------------------------------
# 6. Data Quality Issues Summary
# ---------------------------------------------------
print("\n6. DATA QUALITY ISSUES SUMMARY")
print("-" * 80)

quality_summary = result.agg(
    spark_sum(when(col("unit_price_source") == "cost_price", 1).otherwise(0)).alias("null_prices_fixed"),
    spark_sum(when(col("is_return") == True, 1).otherwise(0)).alias("return_transactions"),
    spark_sum(when(col("discount_source") == "payment_calculated", 1).otherwise(0)).alias("discounts_calculated"),
    spark_sum(when(col("discount_source") == "default_zero", 1).otherwise(0)).alias("discounts_defaulted"),
    spark_sum(when(col("exchange_rate_available") == False, 1).otherwise(0)).alias("missing_exchange_rates")
).collect()[0]

print(f"\nNull Unit Prices Fixed: {quality_summary['null_prices_fixed']:,} ({quality_summary['null_prices_fixed']/total_records*100:.2f}%)")
print(f"Return Transactions: {quality_summary['return_transactions']:,} ({quality_summary['return_transactions']/total_records*100:.2f}%)")
print(f"Discounts Calculated from Payment: {quality_summary['discounts_calculated']:,} ({quality_summary['discounts_calculated']/total_records*100:.2f}%)")
print(f"Discounts Defaulted to Zero: {quality_summary['discounts_defaulted']:,} ({quality_summary['discounts_defaulted']/total_records*100:.2f}%)")
print(f"Missing Exchange Rates: {quality_summary['missing_exchange_rates']:,} ({quality_summary['missing_exchange_rates']/total_records*100:.2f}%)")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)

In [0]:
# =============================================================================
# CREATE CLEAN SILVER LAYER TABLE - ORIGINAL SCHEMA ONLY
# =============================================================================

# Final table with ONLY original columns, cleaned values:
# - invoice_line_id, invoice_id, product_id, quantity, unit_price, discount
# All data quality issues resolved:
# - NULL unit_price → filled with cost_price from products
# - Negative quantity → kept as-is (returns)
# - NULL discount → calculated from payment or defaulted to 0

silver_invoice_line_items = result.select(
    "invoice_line_id",
    "invoice_id",
    "product_id",
    col("quantity_clean").alias("quantity"),
    col("unit_price_clean").alias("unit_price"),
    col("discount_clean").alias("discount")
)

print("Silver Layer Table Created Successfully!")
print(f"Total Records: {silver_invoice_line_items.count():,}")
print("\nCleaned invoice_line_items with original schema:")
display(silver_invoice_line_items.orderBy("invoice_id", "invoice_line_id").limit(50))

# Uncomment below to save to silver layer
# silver_invoice_line_items.write.mode("overwrite").saveAsTable(
#     "silver_catalog.silver_schema.invoice_line_items"
# )
# print("\nTable saved to: silver_catalog.silver_schema.invoice_line_items")